# Open System One models on Colab (JevLite + Kev-4B)

Use this notebook to run the two open checkpoints that can stand next to hosted Jev.

**Runtime (Pro+):** `Runtime → Change runtime type`

1. Hardware accelerator: **L4 GPU** (A100 is fine and a bit faster; T4 is tight; skip H100/TPU).
2. Turn **High RAM** on.
3. Runtime version: Latest.
4. Save, then **Runtime → Run all**.

Before Run all, upload `colab_bundle.zip` from the local package
(`outputs/benchmark/colab_bundle.zip`) when the upload cell asks, or put that zip
on Drive as `MyDrive/colab_bundle.zip`.

In [ ]:
SMOKE = False  # True = 8 items/task; False = full 14k slice
MODELS = "jevlite,kev4"
print("smoke" if SMOKE else "full", MODELS)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Change runtime type to L4 GPU + High RAM, Save, then re-run."
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3
print(props.name, f"{vram_gb:.1f} GB")
if vram_gb < 15:
    raise RuntimeError("Need at least a T4/L4. This runtime is too small for JevLite + Kev-4B.")

In [ ]:
import os
from getpass import getpass
from pathlib import Path

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or ""
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
if not token:
    token = getpass("Hugging Face token (needed if Qwen/Gemma is gated): ").strip()
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    Path.home().joinpath(".cache/huggingface").mkdir(parents=True, exist_ok=True)
    Path.home().joinpath(".cache/huggingface/token").write_text(token, encoding="utf-8")
    print("HF token set")
else:
    print("No HF token; continuing with public downloads")

In [ ]:
from pathlib import Path
from google.colab import files, drive

BUNDLE = Path("/content/colab_bundle.zip")
DRIVE_CANDIDATES = [
    Path("/content/drive/MyDrive/RecSys/jev_experiments/colab_bundle.zip"),
    Path("/content/drive/MyDrive/colab_bundle.zip"),
]

def resolve_drive_bundle():
    return next((path for path in DRIVE_CANDIDATES if path.exists()), None)

if not BUNDLE.exists() and resolve_drive_bundle() is None:
    if not Path("/content/drive/MyDrive").exists():
        print("Mount Drive from the left Files sidebar (folder icon, then the Drive icon), allow access, then re-run this cell.")
        try:
            drive.mount("/content/drive", force_remount=True)
        except Exception as exc:
            print("Programmatic mount failed:", exc)
DRIVE_BUNDLE = resolve_drive_bundle()
if not BUNDLE.exists() and DRIVE_BUNDLE is not None:
    BUNDLE.write_bytes(DRIVE_BUNDLE.read_bytes())
    print("Copied bundle from", DRIVE_BUNDLE)
if not BUNDLE.exists():
    print("Drive is not mounted. Click Choose Files and pick the local colab_bundle.zip.")
    uploaded = files.upload()
    name = next(iter(uploaded))
    src = Path("/content") / name
    if src.resolve() != BUNDLE.resolve():
        BUNDLE.write_bytes(src.read_bytes())
print("bundle", BUNDLE.exists(), BUNDLE.stat().st_size if BUNDLE.exists() else 0)

In [ ]:
import zipfile
from pathlib import Path

BENCH = Path("/content/system_one_search_benchmark")
BENCH.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile("/content/colab_bundle.zip") as archive:
    archive.extractall(BENCH)
print("extracted", len(list(BENCH.rglob("*"))), "paths")

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["PATH"] = "/root/.local/bin:/usr/local/bin:" + os.environ["PATH"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
if not uv:
    subprocess.run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
    os.environ["PATH"] = str(Path.home() / ".local" / "bin") + ":" + os.environ["PATH"]
    uv = shutil.which("uv") or str(Path.home() / ".local" / "bin" / "uv")
print("uv", uv)
venv = Path("/content/open_models_venv")
py = venv / "bin" / "python"
subprocess.run([uv, "python", "install", "3.12"], check=True)
if not py.exists():
    subprocess.run([uv, "venv", str(venv), "--python", "3.12"], check=True)
subprocess.run([uv, "pip", "install", "--python", str(py), "kev[serve] @ git+https://github.com/jaredpalmer/kev.git"], check=True)
subprocess.run([uv, "pip", "install", "--python", str(py), "--reinstall-package", "torch", "--index-url", "https://download.pytorch.org/whl/cu124", "torch"], check=True)
subprocess.run([uv, "pip", "install", "--python", str(py), "flash-linear-attention==0.5.2", "bitsandbytes", "httpx"], check=True)
print(subprocess.check_output([str(py), "-c", "import sys,torch; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"], text=True))

In [ ]:
from pathlib import Path
import subprocess

JEVLITE = Path("/content/jevlite")
if not (JEVLITE / "serve.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vagmi/jevlite.git", str(JEVLITE)], check=True)
print("jevlite", JEVLITE)

In [ ]:
import os, subprocess
from pathlib import Path

os.chdir("/content/system_one_search_benchmark")
py = "/content/open_models_venv/bin/python"
max_items = "8" if SMOKE else "0"
cmd = [
    py, "colab/run_open_models.py",
    "--python", py,
    "--jevlite-src", "/content/jevlite",
    "--max-items", max_items,
    "--models", MODELS,
]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path
report = Path("/content/system_one_search_benchmark/outputs/benchmark/ZERO_SHOT_RESULTS.md")
print(report.read_text(encoding="utf-8"))

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

out = Path("/content/open_model_results.zip")
root = Path("/content/system_one_search_benchmark/outputs/benchmark")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix in {".md", ".json", ".log"} or "predictions" in path.parts:
            archive.write(path, path.relative_to(root).as_posix())
print("downloading", out, out.stat().st_size)
files.download(str(out))